# 00 · Connect Google Colab Pro & Verify Credit Guardrails (`< 0.05 CU`)

Use this quick diagnostic notebook to verify:
1. Your **Google Colab** kernel connection inside **Antigravity IDE** (`Ctrl+Shift+X` -> install `Google Colab` + `Jupyter` -> `Select Kernel -> Colab`).
2. **Google Drive** checkpoint directory (`/content/drive/MyDrive/PCB_FYP_Experiments`).
3. **Kaggle API** credentials (`kaggle.json` on Drive or Colab Secrets) for 200+ MB/s dataset staging onto `/content/datasets` local NVMe SSD.

> **Credit Tip:** Run this verification on a **CPU** or **T4** runtime first; reserve **L4 (24GB)** exclusively for `colab_01_train_multiproduct_detectors_L4.ipynb`.

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = 'https://github.com/Darren7218/PCB-Defect-Detection-FYP.git'
BRANCH   = 'feat/multiproduct-patchcore-v2'
REPO_DIR = Path('/content/PCB-Defect-Detection-FYP')

if Path('/content').exists():
    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))

import torch
print('Working dir :', Path.cwd())
print('PyTorch     :', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device  :', torch.cuda.get_device_name(0))

In [ ]:
from src.cloud.colab_guard import ColabCreditGuard, mount_google_drive

drive_root = mount_google_drive()
exp_dir = (drive_root / 'PCB_FYP_Experiments') if drive_root else Path('./local_experiments')
(exp_dir / 'weights').mkdir(parents=True, exist_ok=True)
(exp_dir / 'datasets').mkdir(parents=True, exist_ok=True)
print('Experiment Sync Root:', exp_dir)

# Configure Kaggle CLI credentials from Google Drive `PCB_FYP_Experiments/kaggle.json` if present
kaggle_src = exp_dir / 'kaggle.json'
if kaggle_src.exists():
    kaggle_dst = Path.home() / '.kaggle' / 'kaggle.json'
    kaggle_dst.parent.mkdir(parents=True, exist_ok=True)
    kaggle_dst.write_bytes(kaggle_src.read_bytes())
    os.chmod(kaggle_dst, 0o600)
    print('Configured ~/.kaggle/kaggle.json from Google Drive!')
else:
    print('Tip: Copy your kaggle.json into', kaggle_src, 'for 200MB/s dataset pulls on Colab.')

In [ ]:
# Smoke-test ColabCreditGuard (auto_unassign=False for interactive check)
with ColabCreditGuard(
    job_name='verify_colab_connection',
    drive_sync_dir=exp_dir / 'logs',
    max_cu_budget=1.0,
    auto_unassign=False,
) as guard:
    print('Guard active! Estimated hourly rate:', guard._cu_rate, 'CU/hr')